# Dependency Parsing — the flow, step by step (BASIC)

A gentle, run-it-and-watch companion to the Session 10 explainer. **Open the explainer
first** (`notes/dependency-parsing/explainer.html`) — this notebook is the same little
machine, written in plain Python so you can *watch* it run.

> This is the **basic** notebook: no classes, no oracle logic, no maths — just three
> lists (`stack`, `buffer`, `arcs`) and four tiny moves. Once this feels easy, the full
> from-scratch version lives in [`arc_eager_parser.ipynb`](arc_eager_parser.ipynb).

**How to use it:** press `Shift`+`Enter` on each cell top to bottom. Read the markdown,
look at the printed output, *then* glance at the code to connect the two. Nothing breaks —
you can always re-run.

## 1 — The sentence and the tree we want

Our sentence is **"He sent her a letter ."** — the full stop is its own token.

Dependency parsing draws an **arrow from each word to the one word it depends on**
(`head -> dependent`). This is the **org chart** we're trying to build — the *answer key*
(the "gold tree"):

```
ROOT   -> sent    (root)    the sentence hangs off the verb
sent   -> He      (nsubj)   who did the sending?
sent   -> her     (iobj)    sent to whom?
sent   -> letter  (dobj)    sent what?
letter -> a       (det)     'a' belongs to 'letter'
sent   -> .       (punct)   the full stop hangs off the verb
```

Every word has exactly **one** head (one manager); `ROOT` is a fake node at the very top
so even the main verb `sent` has a boss.

In [ ]:
# The words. Index 0 is the artificial ROOT; the real sentence starts at index 1.
tokens = ["ROOT", "He", "sent", "her", "a", "letter", "."]

# The gold tree (the answer key): dependent -> (head, label)
gold = {
    "sent":   ("ROOT",   "root"),
    "He":     ("sent",   "nsubj"),
    "her":    ("sent",   "iobj"),
    "letter": ("sent",   "dobj"),
    "a":      ("letter", "det"),
    ".":      ("sent",   "punct"),
}

print("Sentence:", " ".join(tokens[1:]))
print("We want", len(gold), "arrows.")

## 2 — The parser's whole memory: three lists

The parser reads left-to-right and remembers only three things (together: the **configuration**):

| List | What it holds | Starts as |
|------|----------------|-----------|
| `stack`  | words we're still working on (top = **last** item) | `["ROOT"]` |
| `buffer` | words not read yet (front = **first** item)        | the whole sentence |
| `arcs`   | arrows committed so far                            | `[]` (empty) |

We'll always call the two words a move looks at:
- **`s`** = `stack[-1]` — top of the stack
- **`b`** = `buffer[0]` — front of the buffer

When the **buffer is empty**, we're done.

In [ ]:
def new_state():
    """Return a fresh (stack, buffer, arcs) for our sentence."""
    stack  = ["ROOT"]              # top of stack = last element
    buffer = tokens[1:]           # front of buffer = first element (skip ROOT)
    arcs   = []                   # each arc is (head, label, dependent)
    return stack, buffer, arcs

def show(stack, buffer):
    """Print the current stack and buffer in a readable way."""
    print("   stack :", stack, "   <- top is on the right")
    print("   buffer:", buffer, "   <- front is on the left")

stack, buffer, arcs = new_state()
print("Starting configuration:")
show(stack, buffer)
print("   arcs  :", arcs)

## 3 — The four moves, as four tiny functions

This is the heart of it. Each move just edits the three lists:

- **SHIFT** — take `b` off the buffer, put it on the stack (*"read a word in"*).
- **LEFT-ARC(l)** — add arrow `b -> s`, then **pop** `s` (attach a left child, remove it).
- **RIGHT-ARC(l)** — add arrow `s -> b`, then **push** `b` onto the stack (attach a right child, keep it).
- **REDUCE** — **pop** `s` (*"I'm done with this word"*).

Notice the pattern: **SHIFT / RIGHT-ARC** put words *onto* the stack; **LEFT-ARC / REDUCE**
take words *off*.

In [ ]:
def shift(stack, buffer, arcs):
    b = buffer.pop(0)             # take the front word off the buffer
    stack.append(b)               # push it onto the stack

def left_arc(stack, buffer, arcs, label):
    s = stack.pop()               # take the top word off the stack
    b = buffer[0]                 # front of buffer is its head
    arcs.append((b, label, s))    # arrow: b -> s

def right_arc(stack, buffer, arcs, label):
    s = stack[-1]                 # top of stack is the head
    b = buffer.pop(0)             # take the front word off the buffer
    arcs.append((s, label, b))    # arrow: s -> b
    stack.append(b)               # ... and keep b on the stack

def reduce(stack, buffer, arcs):
    stack.pop()                   # drop the top word; we're finished with it

print("Four moves defined: shift, left_arc, right_arc, reduce")

## 4 — The list of moves (the "script")

How does the parser *know* which move to pick? A trained model (or, for teaching, an
**oracle** that peeks at the answer key) decides at each step. To keep this notebook about
the **flow**, we simply write down the correct sequence of moves for our sentence — the
same 10 steps shown in the explainer — and watch them play out.

> The full [`arc_eager_parser.ipynb`](arc_eager_parser.ipynb) builds this sequence
> automatically from the rules. Here we hand it the script so nothing is hidden.

In [ ]:
# Each step is (move_name, label). label is None when the move needs no label.
script = [
    ("SHIFT",     None),
    ("LEFT-ARC",  "nsubj"),   # sent -> He
    ("RIGHT-ARC", "root"),    # ROOT -> sent
    ("RIGHT-ARC", "iobj"),    # sent -> her
    ("SHIFT",     None),
    ("LEFT-ARC",  "det"),     # letter -> a
    ("REDUCE",    None),      # pop 'her'
    ("RIGHT-ARC", "dobj"),    # sent -> letter
    ("REDUCE",    None),      # pop 'letter'
    ("RIGHT-ARC", "punct"),   # sent -> .
]
print(len(script), "moves in the script.")

## 5 — Run it and watch the flow

Now we loop through the script, apply each move, and print the state after every step.
Follow the `stack` shrinking and growing, the `buffer` draining left-to-right, and the
arrows appearing one by one. **This printout is the whole point of the notebook.**

In [ ]:
stack, buffer, arcs = new_state()   # fresh start

for step, (move, label) in enumerate(script, start=1):
    s = stack[-1]                   # top of stack
    b = buffer[0] if buffer else "(empty)"
    labeltxt = "(" + label + ")" if label else ""
    print(f"Step {step:>2}:  s = {s!r:<8} b = {b!r:<8} ->  {move} {labeltxt}")

    if   move == "SHIFT":     shift(stack, buffer, arcs)
    elif move == "LEFT-ARC":  left_arc(stack, buffer, arcs, label)
    elif move == "RIGHT-ARC": right_arc(stack, buffer, arcs, label)
    elif move == "REDUCE":    reduce(stack, buffer, arcs)

    show(stack, buffer)
    if arcs:
        h, l, d = arcs[-1]
        print(f"   NEW arc: {h} -> {d}  ({l})")
    print()

print("Buffer empty ->", len(buffer) == 0, "  ... parsing finished!")

## 6 — Did we build the right tree?

A correct run must reproduce **every** gold arrow and invent none. Let's compare the arcs
we produced against the answer key from step 1.

In [ ]:
produced = {(h, l, d) for (h, l, d) in arcs}
expected = {(head, label, dep) for dep, (head, label) in gold.items()}

print("Produced", len(produced), "arcs, expected", len(expected), "arcs.")
print("Perfect match:", produced == expected)

missing = expected - produced
extra   = produced - expected
if missing: print("Missing:", missing)
if extra:   print("Extra:  ", extra)

## 7 — See the org chart

The `arcs` list is just a bag of arrows. Grouping them by head prints the tree as an
org chart — exactly the picture from the top of the explainer.

In [ ]:
from collections import defaultdict

children = defaultdict(list)
for head, label, dep in arcs:
    children[head].append((dep, label))

def draw(word, indent=0):
    for dep, label in children[word]:
        print("   " * indent + f"-> {dep}  ({label})")
        draw(dep, indent + 1)

print("ROOT")
draw("ROOT")

## 8 — Your turn (poke it!)

The best way to learn is to change one thing and re-run. Try:

1. **Add a `print(arcs)`** at the bottom of the step-5 loop to see the full arc list grow.
2. **Break the script on purpose** — delete the `("REDUCE", None)` on line for popping
   `letter` and re-run steps 5–6. The final `match` becomes `False`. Why? (`.` can't reach
   `sent` while `letter` is blocking the stack top.)
3. **Count the moves:** we used 10 moves for 6 real words. The rule is *at most* `2n`
   moves — here `n = 6`, so `2n = 12`. Confirm `len(script) <= 2 * 6`.

When this notebook feels obvious, open [`arc_eager_parser.ipynb`](arc_eager_parser.ipynb):
it replaces our hand-written `script` with an **oracle** that figures out each move itself.

In [ ]:
# Quick check for tip #3 above:
n = len(tokens) - 1          # real words (exclude ROOT)
print("real words n =", n)
print("moves used   =", len(script))
print("within 2n?   =", len(script) <= 2 * n)